# Práctica: Clasificación con K-Nearest Neighbors (KNN)
## Módulo: MP4 Sistemes de Big Data - RA1
### Dataset: `samybaladram/iris-dataset-extended` (Kaggle)

---
### Objetivos de la Fase 1 (Python):
1. **Cargar el dataset y mostrar su estructura.**
2. **Describir los tipos de datos y la representación de la clase.**
3. **Preparar la partición y el escalado de características.**
4. **Entrenar y evaluar el modelo KNN.**
5. **Generar una visualización completa de resultados y fronteras.**
6. **Identificar qué partes del código dependen de bibliotecas externas vs. Python nativo.**


### 0. Instalación e importación de dependencias
Descomenta la celda inferior si ejecutas este cuaderno en **Google Colab**.

In [ ]:
# !pip install "kagglehub[pandas-datasets]" scikit-learn seaborn matplotlib pandas numpy

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA

# Kagglehub
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Configuración estética
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


---
## 1. Cargar el dataset y mostrar su estructura
Cargamos el dataset extendido de Iris desde Kaggle utilizando `kagglehub`. En este repositorio existen dos archivos: `iris_original.csv` e `iris_extended.csv`. Especificamos `file_path = 'iris_extended.csv'`.


In [ ]:
# Ruta al archivo específico del dataset dentro de KaggleHub
file_path = "iris_extended.csv"

# Intentar carga vía KaggleHub o fallback a archivo local
if os.path.exists(file_path):
    print(f"Cargando desde copia local: {file_path}")
    df = pd.read_csv(file_path)
else:
    print(f"Descargando vía kagglehub: samybaladram/iris-dataset-extended/{file_path}")
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "samybaladram/iris-dataset-extended",
        file_path
    )

print("Shape del DataFrame (filas, columnas):", df.shape)
print("\nPrimeros 5 registros:")
display(df.head())

print("\nEstructura de tipos y valores nulos:")
df.info()


---
## 2. Describir los tipos de datos y la representación de la clase
Analizamos:
- **Variable objetivo (`species`)**: cardinalidad, balance de clases y porcentaje.
- **Variables predictoras numéricas**: 19 variables continuas (longitudes, áreas, ratios morfométricos).
- **Variables predictoras categóricas**: `soil_type` (tipo de suelo).
- **Valores nulos / faltantes**.


In [1]:
# 1. Análisis de la variable objetivo (clase)
target_col = "species"
conteo = df[target_col].value_counts()
porcentaje = df[target_col].value_counts(normalize=True) * 100
balance_df = pd.DataFrame({"Muestras": conteo, "Porcentaje (%)": porcentaje})
print("--- Distribución de la Clase (Target) ---")
display(balance_df)

# 2. Análisis de tipos de variables
num_cols = df.drop(columns=[target_col]).select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.drop(columns=[target_col]).select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\nVariables Numéricas ({len(num_cols)}):", num_cols)
print(f"Variables Categóricas ({len(cat_cols)}):", cat_cols)

for c in cat_cols:
    print(f"Valores únicos en '{c}':", df[c].unique().tolist())

# 3. Estadísticas descriptivas de los atributos numéricos
display(df[num_cols].describe().T[["mean", "std", "min", "50%", "max"]].head(10))


NameError: name 'df' is not defined

---
## 3. Preparar la partición y el escalado

### Consideraciones clave:
1. **Codificación One-Hot para categóricas**: `soil_type` se convierte a variables binarias (`dummy variables`).
2. **Codificación de etiqueta**: `species` se transforma a enteros $\{0, 1, 2\}$ con `LabelEncoder`.
3. **Partición estratificada (Train/Test)**: 80% entrenamiento (960 muestras) y 20% prueba (240 muestras), manteniendo la proporción balanceada de cada clase.
4. **Escalado con `StandardScaler`**:
   - **¿Por qué es fundamental en KNN?** KNN calcula distancias euclidianas: $d(p, q) = \sqrt{\sum (p_i - q_i)^2}$. Variables de gran magnitud (ej. `elevation` $\approx 100-300$) dominarían por completo sobre variables con rangos pequeños (ej. `petal_width` $\approx 0.1-2.5$).
   - **Prevención de Data Leakage**: Se aplica `fit_transform` exclusivamente sobre el conjunto de entrenamiento (`X_train`), y únicamente `transform` sobre el conjunto de prueba (`X_test`).


In [ ]:
# Separar características predictoras (X) y objetivo (y)
X = df.drop(columns=[target_col])
y = df[target_col]

# One-Hot Encoding de variables categóricas
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Codificación de la clase a valores numéricos
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Mapeo de clases:", dict(zip(le.classes_, range(len(le.classes_)))))

# Partición Estratificada 80% Train - 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print(f"Dimensiones X_train: {X_train.shape} | X_test: {X_test.shape}")

# Escalado con StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Media en Train escalado:", np.round(X_train_scaled.mean(), 4))
print("Desviación estándar en Train escalado:", np.round(X_train_scaled.std(), 4))


---
## 4. Entrenar y evaluar KNN

Optimización del hiperparámetro $k$ mediante validación cruzada estratificada de 5 pliegues (`StratifiedKFold`), seguido del entrenamiento y evaluación del modelo final.


In [ ]:
# Búsqueda del mejor k mediante validación cruzada
k_range = range(1, 31)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
cv_stds = []

for k in k_range:
    knn_cv = KNeighborsClassifier(n_neighbors=k, metric='minkowski', p=2)
    scores = cross_val_score(knn_cv, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    cv_scores.append(scores.mean())
    cv_stds.append(scores.std())

cv_scores = np.array(cv_scores)
best_k = list(k_range)[np.argmax(cv_scores)]
print(f"Mejor hiperparámetro: k = {best_k} (Accuracy CV: {cv_scores.max()*100:.2f}%)")

# Entrenar modelo final con k óptimo
knn = KNeighborsClassifier(n_neighbors=best_k, metric='minkowski', p=2)
knn.fit(X_train_scaled, y_train)

# Predicción en el conjunto de prueba no visto
y_pred = knn.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\nExactitud en el Conjunto de Test: {test_acc * 100:.2f}%")
print("\n--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("--- Matriz de Confusión ---")
display(pd.DataFrame(cm, index=[f"Real_{c}" for c in le.classes_], columns=[f"Pred_{c}" for c in le.classes_]))


---
## 5. Visualización de resultados y fronteras de decisión

Generamos una figura completa de 4 cuadrantes que sintetiza el comportamiento del clasificador:
1. **Matriz de Confusión Normalizada** (Mapa de calor).
2. **Curva de Rendimiento según $k$** (Con intervalo de desviación estándar).
3. **Fronteras de Decisión en 2D** (Proyección PCA para visualizar el espacio de decisión).
4. **Dispersión de Atributos Clave** (`petal_length` vs `petal_width`).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
plt.subplots_adjust(wspace=0.3, hspace=0.35)

class_names = le.classes_

# 1. Matriz de Confusión
ax1 = axes[0, 0]
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(
    cm_norm, annot=True, fmt=".2%", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names, ax=ax1,
    linewidths=1.5, annot_kws={"size": 12, "weight": "bold"}
)
ax1.set_title(f"1. Matriz de Confusión (Test Set, k={best_k})", fontsize=13, fontweight='bold', pad=10)
ax1.set_xlabel("Clase Predicha", fontsize=11)
ax1.set_ylabel("Clase Real", fontsize=11)

# 2. Curva de Optimización de k
ax2 = axes[0, 1]
ax2.plot(list(k_range), cv_scores * 100, marker='o', color='#1f77b4', linewidth=2, label="Accuracy Media CV")
ax2.fill_between(
    list(k_range), (cv_scores - cv_stds) * 100, (cv_scores + cv_stds) * 100,
    color='#1f77b4', alpha=0.2, label="± 1 Desviación Típica"
)
ax2.axvline(x=best_k, color='crimson', linestyle='--', linewidth=2, label=f"Mejor k = {best_k}")
ax2.scatter([best_k], [cv_scores[list(k_range).index(best_k)] * 100], color='crimson', s=120, zorder=5)
ax2.set_title("2. Curva de Optimización de Hiperparámetro k", fontsize=13, fontweight='bold', pad=10)
ax2.set_xlabel("Número de Vecinos (k)", fontsize=11)
ax2.set_ylabel("Exactitud (%) en CV", fontsize=11)
ax2.legend(loc="lower left")
ax2.grid(True, linestyle=":", alpha=0.6)

# 3. Fronteras de Decisión en 2D vía PCA
ax3 = axes[1, 0]
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

knn_pca = KNeighborsClassifier(n_neighbors=best_k)
knn_pca.fit(X_train_pca, y_train)

h = 0.05
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = knn_pca.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

from matplotlib.colors import ListedColormap
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
cmap_bold = ['#d62728', '#2ca02c', '#1f77b4']

ax3.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.5)
for idx, c_name in enumerate(class_names):
    mask = (y_test == idx)
    ax3.scatter(
        X_test_pca[mask, 0], X_test_pca[mask, 1],
        c=cmap_bold[idx], label=f"Real: {c_name}",
        edgecolor='k', s=40, alpha=0.85
    )
var_exp = pca.explained_variance_ratio_ * 100
ax3.set_title(f"3. Fronteras de Decisión KNN (PCA 2D: {var_exp.sum():.1f}% Varianza)", fontsize=13, fontweight='bold', pad=10)
ax3.set_xlabel(f"PC1 ({var_exp[0]:.1f}%)", fontsize=11)
ax3.set_ylabel(f"PC2 ({var_exp[1]:.1f}%)", fontsize=11)
ax3.legend(loc="upper right", fontsize=9)
ax3.grid(True, linestyle=":", alpha=0.4)

# 4. Dispersión Petal Length vs Petal Width
ax4 = axes[1, 1]
sns.scatterplot(
    data=df, x="petal_length", y="petal_width", hue="species",
    palette=cmap_bold, s=45, alpha=0.8, edgecolor='k', ax=ax4
)
ax4.set_title("4. Separabilidad de Clases (Petal Length vs Petal Width)", fontsize=13, fontweight='bold', pad=10)
ax4.set_xlabel("Petal Length (cm)", fontsize=11)
ax4.set_ylabel("Petal Width (cm)", fontsize=11)
ax4.legend(title="Especie", loc="upper left")
ax4.grid(True, linestyle=":", alpha=0.5)

plt.show()


---
## 6. Identificación de dependencias de bibliotecas externas vs. Python nativo

| Biblioteca / Módulo | Funcionalidades Específicas Utilizadas | Responsabilidad Principal |
| :--- | :--- | :--- |
| **`kagglehub`** | `load_dataset`, `KaggleDatasetAdapter` | Descarga y caché automatizada del dataset directamente desde la plataforma Kaggle. |
| **`pandas`** | `read_csv`, `DataFrame`, `info`, `describe`, `get_dummies`, `value_counts` | Manipulación de estructuras tabulares bidimensionales, One-Hot Encoding e inspección exploratoria. |
| **`numpy`** | `array`, `meshgrid`, `c_`, `mean`, `std`, `argmax`, `trace` | Operaciones vectorizadas y matriciales de alto rendimiento, creación de mallas 2D. |
| **`scikit-learn`** (`sklearn.model_selection`) | `train_test_split`, `StratifiedKFold`, `cross_val_score` | Particionado estratificado de datos y validación cruzada para ajuste de hiperparámetros. |
| **`scikit-learn`** (`sklearn.preprocessing`) | `StandardScaler`, `LabelEncoder` | Normalización/estandarización z-score ($z = \frac{x-\mu}{\sigma}$) y codificación de clases nominales. |
| **`scikit-learn`** (`sklearn.neighbors`) | `KNeighborsClassifier` | Implementación optimizada del algoritmo KNN (con árboles espaciales KD-Tree / Ball-Tree para búsqueda rápida $O(k \log N)$). |
| **`scikit-learn`** (`sklearn.metrics`) | `accuracy_score`, `classification_report`, `confusion_matrix` | Evaluación cuantitativa del clasificador (matriz de confusión, precisión, exhaustividad y F1-score). |
| **`scikit-learn`** (`sklearn.decomposition`) | `PCA` | Reducción lineal de dimensionalidad de 21 a 2 variables para proyección geométrica 2D. |
| **`matplotlib.pyplot`** | `subplots`, `plot`, `scatter`, `contourf`, `savefig` | Motor de bajo nivel para trazado de figuras, ejes, contornos de decisión y curvas de ajuste. |
| **`seaborn`** | `heatmap`, `scatterplot`, `set_theme` | Capa de visualización estadística de alto nivel para matrices de confusión estéticas y dispersiones multicolores. |

### Componentes de Python Nativo (Built-in)
- **Estructuras de datos nativas**: Listas (`[c for c in cols]`), diccionarios (`dict(zip(...))`), tuplas y rangos numéricos (`range(1, 31)`).
- **Control de flujo**: Sentencias condicionales `if/else`, bucles `for`, manejadores de excepciones `try/except`.
- **Librería estándar**: `os` y `sys` para inspección de rutas y detección del sistema operativo.
